# Mini Project: Retail Data Analysis with Seaborn

This project uses all 8 retail CSV files and focuses on Seaborn visualizations. It includes six Seaborn-specific plots with features such as `hue`, `size`, `style`, statistical summaries, distributions, and relationships.

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re

sns.set_theme(style="darkgrid", palette="viridis", context="notebook")

## Section 2: Upload All 8 CSV Files

Select all eight files together. The code accepts filenames with or without `(1)`.

In [ ]:
from google.colab import files

uploaded = files.upload()

def normalize_name(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    name = re.sub(r"\\s*\\(\\d+\\)", "", name)
    return re.sub(r"[^a-z0-9]", "", name.lower())

required_names = [
    "targets", "stores", "returns", "products",
    "orders", "order_items", "employees", "customers"
]

file_map = {}
for required in required_names:
    matches = [f for f in uploaded.keys()
               if normalize_name(f) == normalize_name(required)]
    if matches:
        file_map[required] = matches[0]

missing = [name for name in required_names if name not in file_map]
if missing:
    raise FileNotFoundError(f"Upload these missing CSV files: {missing}")

print("All 8 CSV files uploaded successfully!")

## Section 3: Read All Datasets

In [ ]:
targets = pd.read_csv(file_map["targets"])
stores = pd.read_csv(file_map["stores"])
returns = pd.read_csv(file_map["returns"])
products = pd.read_csv(file_map["products"])
orders = pd.read_csv(file_map["orders"])
order_items = pd.read_csv(file_map["order_items"])
employees = pd.read_csv(file_map["employees"])
customers = pd.read_csv(file_map["customers"])

print("All datasets loaded.")
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)
print("Employees:", employees.shape)
print("Returns:", returns.shape)
print("Targets:", targets.shape)

## Section 4: Combine the Data and Calculate Sales

In [ ]:
# Convert date columns
for df, column in [
    (orders, "OrderDate"),
    (products, "LaunchDate"),
    (returns, "ReturnDate"),
    (employees, "HireDate"),
    (customers, "SignupDate")
]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

# Rename overlapping columns before merging
customer_info = customers.rename(columns={"Name": "CustomerName", "City": "CustomerCity"})
employee_info = employees.rename(columns={"Name": "EmployeeName", "Region": "EmployeeRegion"})
store_info = stores.rename(columns={"Region": "StoreRegion", "City": "StoreCity"})

data = order_items.merge(orders, on="OrderID", how="left")
data = data.merge(products, on="ProductID", how="left")
data = data.merge(customer_info, on="CustomerID", how="left")
data = data.merge(store_info, on="StoreID", how="left")
data = data.merge(employee_info, on="EmployeeID", how="left")
data = data.merge(returns, on="OrderID", how="left")

data["Sales"] = data["Quantity"] * data["Price"] * (1 - data["Discount"])
data["Profit"] = data["Quantity"] * (data["Price"] * (1 - data["Discount"]) - data["Cost"])
data["Month"] = data["OrderDate"].dt.to_period("M").astype("string")

print("Combined data shape:", data.shape)
data.head()

## Section 5: Seaborn Graph 1 — Scatter Plot with Hue and Size
Shows the relationship between quantity and sales. `hue` separates product categories and `size` represents discount.

In [ ]:
plot_data = data.dropna(subset=["Quantity", "Sales", "Category", "Discount"])

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=plot_data,
    x="Quantity",
    y="Sales",
    hue="Category",
    size="Discount",
    sizes=(30, 180),
    alpha=0.7
)
plt.title("Quantity vs Sales by Product Category")
plt.tight_layout()
plt.show()

## Section 6: Seaborn Graph 2 — Violin Plot
Shows the distribution and density of sales for each product category.

In [ ]:
plot_data = data.dropna(subset=["Category", "Sales"])

plt.figure(figsize=(9, 5))
sns.violinplot(
    data=plot_data,
    x="Category",
    y="Sales",
    inner="quartile",
    cut=0
)
plt.title("Sales Distribution Across Product Categories")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Section 7: Seaborn Graph 3 — Heatmap
Displays correlations among numeric sales-related columns.

In [ ]:
numeric_columns = ["Quantity", "Discount", "Price", "Cost", "Sales", "Profit"]
correlation = data[numeric_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Heatmap of Retail Data")
plt.tight_layout()
plt.show()

## Section 8: Seaborn Graph 4 — Pair Plot
Compares multiple numeric variables together. A sample is used to keep the graph quick to render.

In [ ]:
pair_data = data[["Quantity", "Price", "Discount", "Sales", "Category"]].dropna()
pair_data = pair_data.sample(n=min(400, len(pair_data)), random_state=42)

sns.pairplot(
    pair_data,
    vars=["Quantity", "Price", "Discount", "Sales"],
    hue="Category",
    diag_kind="hist",
    corner=True
)
plt.show()

## Section 9: Seaborn Graph 5 — Joint Plot with Regression
Shows the relationship between product price and calculated sales, including a regression line and marginal distributions.

In [ ]:
joint_data = data[["Price", "Sales"]].dropna()
joint_data = joint_data.sample(n=min(1000, len(joint_data)), random_state=42)

sns.jointplot(
    data=joint_data,
    x="Price",
    y="Sales",
    kind="reg",
    height=7,
    scatter_kws={"alpha": 0.35}
)
plt.suptitle("Product Price vs Sales", y=1.02)
plt.show()

## Section 10: Seaborn Graph 6 — Count Plot with Hue
Compares order status across sales channels.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(
    data=orders.dropna(subset=["SalesChannel", "OrderStatus"]),
    x="SalesChannel",
    hue="OrderStatus"
)
plt.title("Order Status by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Number of Orders")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## Section 11: Bonus — Monthly Sales Trend with Seaborn

In [ ]:
monthly_sales = (
    data.dropna(subset=["Month"])
        .groupby("Month", as_index=False)["Sales"]
        .sum()
        .sort_values("Month")
)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=monthly_sales,
    x="Month",
    y="Sales",
    marker="o",
    linewidth=2.5
)
plt.title("Monthly Retail Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Conclusion

This mini project uses all eight retail CSV files and demonstrates Seaborn's statistical and relational visualizations: scatter plot, violin plot, heatmap, pair plot, joint plot, count plot, and line plot.